In [1]:
#import relevant libraries: pip install re, pip install natsort, pip install plotly==5.10.0
import sys
import os
import glob

import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import datetime as dt8
import math
import matplotlib.pyplot as plt
import decimal
import re
from natsort import index_natsorted
import dabest

import NLCLIMB 
import NLMATH

import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#NOTE: SUPPRESSES WARNINGS!

import warnings

warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 42.82it/s]

Numba compilation complete!


In [2]:
#Initial file processing
computer1 = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data\\"
openPath = computer1 + filedir

files = os.listdir(openPath)
filename = openPath + "Fecundity.csv"

dfe=pd.read_csv(filename)

In [3]:
df = dfe.iloc[0:8,:]


In [4]:
df

,Genotype,Day of plate collection,Number of embroys laid,Pupa collected,Pupated (/10):,Eclosed:,Proportion pupated:,Proportion eclosed:
0,w1118 x elav,3/7/2025,17.0,10.0,4.0,4.0,0.4,1.0
1,elav x eOPN3,3/7/2025,5.0,5.0,4.0,4.0,0.8,1.0
2,elav x ACR,3/7/2025,24.0,10.0,0.0,0.0,0.0,0.0
3,w1118 x elav,3/8/2025,14.0,10.0,2.0,2.0,0.2,1.0
4,elav x eOPN3,3/8/2025,11.0,10.0,5.0,5.0,0.5,1.0
5,elav x ACR,3/8/2025,44.0,10.0,0.0,0.0,0.0,0.0
6,w1118 x elav,3/9/2025,21.0,10.0,2.0,2.0,0.2,2.0
7,elav x ACR,3/9/2025,13.0,10.0,0.0,0.0,0.0,0.0


In [18]:
dfpupate = pd.DataFrame()
dfeclose = pd.DataFrame()
for n in df['Genotype'].unique():
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()
    pupacollected = int(df[df['Genotype'] == n]['Pupa collected'].sum())
    pupated = int(df[df['Genotype'] == n]['Pupated (/10):'].sum())
    eclosed = int(df[df['Genotype'] == n]['Eclosed:'].sum())
    failed = pupacollected-pupated                        
    df1[n + "_pupate"] = [0]*failed + [1]*pupated
    df1 = df1.reset_index(drop = True)
    
    df2[n + "_eclose"] = [1]*pupated + [0]*(pupated-eclosed)
    df2 = df2.reset_index(drop = True)
    dfeclose = pd.concat([dfeclose, df2], axis =1)
    dfpupate = pd.concat([dfpupate, df1], axis =1)

dftotal = pd.concat([dfpupate, dfeclose], axis =1).reset_index(drop = True)

print(dftotal)

    w1118 x elav_pupate  elav x eOPN3_pupate  elav x ACR_pupate  \
0                     0                  0.0                  0   
1                     0                  0.0                  0   
2                     0                  0.0                  0   
3                     0                  0.0                  0   
4                     0                  0.0                  0   
5                     0                  0.0                  0   
6                     0                  1.0                  0   
7                     0                  1.0                  0   
8                     0                  1.0                  0   
9                     0                  1.0                  0   
10                    0                  1.0                  0   
11                    0                  1.0                  0   
12                    0                  1.0                  0   
13                    0                  1.0                  

In [ ]:
paired_prop2 = dabest.load(data = dftotal, proportional=True, 
                           id_col="index", paired='baseline', 
                           x = ["ExperimentState", "ExperimentState"], 
                           y = "value", delta2=True,
                           experiment="Type",
                           x1_level=['Dark', 'Full'],
                           experiment_label=['WT', driver])